<h1>🎛️ Biofilter — Report: <code>expand_gene_to_variant</code></h1>

The variants belonging to a list of genes, with their annotation.

"Belonging to" is **two different questions**, and this report makes you
pick one. Section 3 is about why that is not a detail.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

BUNDLE = None
REPORT = "expand_gene_to_variant"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(bf.core.db_uri)

### 2. Which chromosomes does this bundle carry

Ask first. The current bundle is chr22 only, so a gene anywhere else
resolves fine and then finds nothing — which is a property of the build,
not of the gene.

In [ ]:
from biofilter.modules.report import Bundle

with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    chroms = bundle.chromosomes("variant_masters")

print("chromosomes with variants:", chroms)

### 3. The choice you have to make

| `mapping` | a variant belongs to a gene when… |
| --- | --- |
| `position` | its coordinate falls inside the gene's build-38 range |
| `annotation` | VEP associated it with that gene |

Run the same gene both ways and compare.

In [ ]:
GENE = "CHEK2"

sets = {}
for mapping in ("position", "annotation"):
    out = bf.report.run(REPORT, input_data=[GENE], mapping=mapping,
                        max_variants_per_gene=0).to_pandas()
    sets[mapping] = set(out.query("status == 'ok'").variant_key)
    print(f"  {mapping:<11} {len(sets[mapping]):>6,} variants")

p, a = sets["position"], sets["annotation"]
print(f"\n  both         {len(p & a):>6,}")
print(f"  only position{len(p - a):>7,}")
print(f"  only annotation{len(a - p):>5,}")

For `CHEK2` the positional set turns out to be a clean subset of the
annotated one — VEP assigns every in-body variant to the gene *and*
reaches about 5 kb beyond it. That is common but not universal.

Across all **958 chr22 genes** with build-38 coordinates and annotated
variants:

| | pairs |
| --- | --- |
| by position | 2,045,943 |
| by annotation | 2,651,135 |
| **only by position** | **144,488** (139 genes have at least one) |
| only by annotation | ~749,680 |

Only-position variants are the ones VEP attributed to a neighbouring
gene, or to no gene at all. So neither mechanism contains the other, and
nothing in the rows tells you which one you ran.

### 4. So the report records the choice — twice

Once as a column on every row, once in the provenance JSON. A CSV that
gets separated from its provenance file still says which question it
answers.

In [ ]:
result = bf.report.run(REPORT, input_data=[GENE], mapping="position")
df = result.to_pandas()

print("column :", df["mapping"].unique().tolist())
print("provenance:")
result.provenance["mapping"]

### 5. `window_bp` — widening the gene's range

Applies to `position` only. Passing it with `mapping="annotation"` is an
error rather than a silent no-op: VEP's own association already reaches
past the gene body, so a window there would change nothing while looking
like it had.

In [ ]:
for window in (0, 5_000, 50_000):
    out = bf.report.run(REPORT, input_data=[GENE], mapping="position",
                        window_bp=window, max_variants_per_gene=0).to_pandas()
    print(f"  window_bp={window:>6,}  {len(out):>7,} rows")

try:
    bf.report.run(REPORT, input_data=[GENE], mapping="annotation", window_bp=5_000)
except ValueError as exc:
    print("\nrefused:", exc)

### 6. The cap, and why it announces itself

`max_variants_per_gene` defaults to 5000. A capped gene looks exactly
like a complete answer — a round number of rows and nothing admitting
more existed — so the provenance says what was hidden, per gene.

In [ ]:
capped = bf.report.run(REPORT, input_data=["CHEK2", "SMARCB1"],
                       mapping="position", max_variants_per_gene=200)

print(capped.to_pandas().groupby("input_gene").size().to_dict())
capped.provenance["truncation"]

In [ ]:
# 0 means no cap — not "fall back to the default".
uncapped = bf.report.run(REPORT, input_data=["CHEK2", "SMARCB1"],
                         mapping="position", max_variants_per_gene=0)

print(uncapped.to_pandas().groupby("input_gene").size().to_dict())
print("applied:", uncapped.provenance["truncation"]["applied"])

### 7. Four statuses, and one distinction that matters

| status | means |
| --- | --- |
| `ok` | a variant |
| `not_found` | the input did not resolve to a gene in this bundle |
| `no_location` | the gene resolved, but the bundle has no build-38 coordinates — `position` cannot place it |
| `no_variants` | the gene resolved and nothing met the criteria |

`no_location` is not a rare corner: **33,354 of the 72,660 genes in this
bundle — 45.9% — have no build-38 coordinates**. For those, `annotation`
is the only mapping that can answer anything.

In [ ]:
mixed = bf.report.run(REPORT, input_data=["CHEK2", "TP53", "NOT_A_GENE"],
                      mapping="position", max_variants_per_gene=50).to_pandas()

mixed.groupby(["input_gene", "status"]).size().to_frame("rows")

In [ ]:
# TP53 is on chr17; this bundle carries chr22 only. The report says
# which of the two reasons applies rather than returning an empty frame.
mixed.query("status != 'ok'")[["input_gene", "status", "note"]]

### 8. One row per variant, or one per transcript

`most_severe_only=True` (the default) collapses a variant's transcripts
and keeps the worst consequence. Turn it off and a row becomes a
gene-variant-**transcript** triple — counting rows then counts
transcripts, not variants.

In [ ]:
for severe in (True, False):
    out = bf.report.run(REPORT, input_data=[GENE], mapping="annotation",
                        most_severe_only=severe, max_variants_per_gene=0).to_pandas()
    print(f"  most_severe_only={str(severe):<5}  "
          f"{len(out):>7,} rows  {out.variant_key.nunique():>6,} variants")

### 9. Filtering

Frequency, impact, consequence and the in-silico predictors. They stack.

In [ ]:
rare_damaging = bf.report.run(
    REPORT,
    input_data=[GENE],
    mapping="position",
    af_max=0.001,
    impact_filter=["HIGH", "MODERATE"],
    cadd_phred_min=20,
    max_variants_per_gene=0,
).to_pandas()

print(f"{len(rare_damaging):,} rows")
rare_damaging[["variant_key", "rsid", "consequence", "impact",
               "af_joint", "cadd_phred", "alphamissense_classification"]].head(8)

### 10. A missing predictor is a missing column, not a zero

If the bundle carries no `variant_predictions` or `variant_alphamissense`
table, those columns come back null for every row. The omission is
recorded in the provenance `coverage` block — check it before concluding
that nothing scored.

In [ ]:
result.provenance["coverage"]

In [ ]:
# AlphaMissense needs two corrections to join, and both failures are
# silent nulls: it versions transcript ids (ENST00000327374.9) where VEP
# does not, and it scores a transcript VEP rarely calls most severe.
#
# So the join follows the grain of the row. One row per variant gets the
# variant's score; one row per transcript gets that transcript's.
for severe in (True, False):
    out = bf.report.run(REPORT, input_data=[GENE], mapping="position",
                        af_max=0.001, impact_filter=["HIGH", "MODERATE"],
                        cadd_phred_min=20, most_severe_only=severe,
                        max_variants_per_gene=0)
    mis = out.to_pandas().query("consequence == 'missense_variant'")
    scored = int(mis["alphamissense_score"].notna().sum())
    print(f"  most_severe_only={str(severe):<5}  {len(mis):>6,} missense, "
          f"{scored:>6,} scored  "
          f"({out.provenance['alphamissense']['joined_on']})")

### 11. Export

In [ ]:
for path in result.write(OUTPUT_DIR / "expand_gene_to_variant.csv"):
    print(path)

### 12. Chaining is your job, on purpose

There is no report-to-report plumbing. Write the list out, look at it,
pass it on.

```python
variants = df.query("status == 'ok'").variant_key.tolist()
bf.report.run("expand_variant_regulatory", input_data=variants)
```

### 13. The same thing on the command line

```bash
biofilter report run --report-name expand_gene_to_variant \\
    --input CHEK2 --input SMARCB1 \\
    --param mapping=position \\
    --param window_bp=5000 \\
    --param af_max=0.01 \\
    --output gene_variants.csv
```